# CS781 Phase 6 · legal abstractive summarization · Kaggle T4 × 2

**Run all cells in order.** This notebook trains six real candidates from your four **original Phase-1** foundations, selects checkpoints by held-out **BERTScore F1**, and writes six individual submission ZIPs plus `best_submission.zip`. The native encoder–decoder with full adaptation is the main score candidate; the final choice is measured on your held-out documents.

No Phase-5 checkpoint is required: Phase-6 rules require every experiment to start from Phase 1. Test references, if present, are discarded. No external pretrained generator, sentence encoder, API generation, or leaderboard-based tuning is used. The pretrained BERTScore network is used **only for evaluation**, matching the official evaluator.

**Performance is unmeasured until you execute this notebook.** Phase-5 Macro-F1 is a different task/metric and does not predict Phase-6 BERTScore. A high leaderboard rank cannot be guaranteed.

Official sources checked November 7, 2026: [capstone brief](https://docs.google.com/document/d/1olqoT5KPhf5CSJLBNjXGyKyyyQDBr6jcnWqPUh5cNk4/edit), [dataset](https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone), [Phase-6 evaluator](https://drive.google.com/file/d/1j2TZtu88BcLeeh9ioVlRuNMdDd_z9gfo/view). The brief contains differing minimum-submission counts; six is the stricter general requirement. Its stated Phase-6 deadline was November 2, 2026.

**Authorship:** AI-assisted implementation draft, with research attribution below. Follow your instructor’s permitted assistance scope; brief §17.5 restricts AI-written architecture and tuning. Do not describe this code as independently authored.

## Repair notes for the uploaded run

The saved run stopped while parsing a training summary, before GPU preflight or model training. The loader now accepts summary strings, lists of text fragments, and explicit nested text fields or reference aliases. Training records without usable text or a reference are excluded with their IDs and field types in `token_cache/audit.json`. Test references stay unused and official test IDs are preserved; invalid test documents produce a named error. The new token cache has its own filename, so previous files and downloaded datasets are retained.

The installation cell now installs missing packages without upper caps that forced downgrades. The two-GPU preflight also uses a conservative FP16 scale with bounded retries for gradient overflow and a separate diagnostic for a non-finite forward loss.

**Existing Kaggle session:** paste the provided repair-cell script above the failed data-preparation cell, run it, then rerun data preparation and continue to preflight and training. **Fresh session:** use this notebook and run from the top. CPU fixtures were checked; the real Kaggle dataset and two-T4 run have not been rerun here.


## Inputs and hardware

Select **GPU T4 × 2** and enable Internet for package/data/metric downloads. Attach your `phase-1-outputs` dataset. Optionally attach the official Phase-6 data; otherwise it is downloaded at a pinned Hugging Face commit. For a gated dataset, accept its terms and add a Kaggle secret named `HF_TOKEN`.

```text
/kaggle/input/datasets/hardik4tiwari/phase-1-outputs/capstone_phase1/
  tokenizer.json
  runs/main_decoder/best_model.pt
  runs/main_encoder_decoder/best_model.pt
  runs/main_local_gqa/best_model.pt
  runs/main_hybrid/best_model.pt

<any attached dataset>/Phase_6_Summ/
  Train/train.json   # id, document, summary
  Test/test.json     # id, document; any summary is ignored
```

The loader checks each original checkpoint’s architecture, tokenizer hash, random-initialization provenance and 50M–300M parameter count. It deliberately does not select task-adapted checkpoints from other phases.

In [ ]:
import os, sys, subprocess, importlib.util
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ.setdefault('HF_HOME', '/kaggle/working/hf_cache_phase6')
# Leave Kaggle's CUDA torch and existing compatible HF stack in place.
requirements = {'tokenizers':'tokenizers', 'huggingface_hub':'huggingface-hub',
                'sklearn':'scikit-learn', 'rouge_score':'rouge-score',
                'transformers':'transformers', 'bert_score':'bert-score'}
missing = [package for module, package in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('Required packages already installed; no dependency changes.')


In [ ]:
from pathlib import Path
import json, shutil, torch
PROJECT = Path('/kaggle/working/capstone_phase6_v1')
PHASE1_ROOT = Path('/kaggle/input/datasets/hardik4tiwari/phase-1-outputs/capstone_phase1')
DATA_ROOT = None       # e.g. '/kaggle/input/<dataset>/Phase_6_Summ'; None auto-detects or downloads.
DATA_REVISION = None   # None resolves once and persists the exact official dataset commit.
WORLD_SIZE = 2
SETTINGS = {
    'seed': 781, 'dev_fraction': 0.08, 'dev_max': 512,
    'source_tokens': 2048, 'target_tokens': 1536,
    # Effective document batch = 1 × 8 × 2 = 16.
    'micro_batch': 1, 'accumulation': 8, 'generation_batch': 2,
    'full_epochs': 6, 'peft_epochs': 6, 'student_epochs': 4,
    'patience': 2, 'dev_probe': 128,
    'decode_tune_cases': 64, 'tune_beams': True,
    # Real teacher targets cached once on up to 1024 TRAIN documents; remaining cases use gold.
    'kd_cases': 1024,
    'special_ids': {},  # Override only with verified existing PAD/BOS/EOS IDs if auto-detection fails.
}
assert torch.cuda.is_available() and torch.cuda.device_count() >= WORLD_SIZE, 'Select GPU T4 × 2.'
PROJECT.mkdir(parents=True, exist_ok=True)
CODE = PROJECT / 'code'; CODE.mkdir(exist_ok=True)
print('PyTorch:', torch.__version__)
for i in range(WORLD_SIZE): print(i, torch.cuda.get_device_name(i), round(torch.cuda.get_device_properties(i).total_memory/2**30, 1), 'GiB')
print('Effective document batch:', WORLD_SIZE * SETTINGS['micro_batch'] * SETTINGS['accumulation'])
print('Free disk GiB:', round(shutil.disk_usage(PROJECT).free/2**30, 1))
# Choose settings before training; resume rejects changed configuration/data/code.


## Architecture and research choices

All runs share a **train-only learned sentence selector**. A hashing-based salience regressor learns lexical reference overlap using training summaries. At inference it sees only the document. Maximal marginal relevance reduces duplicate sentences; small first/last quotas preserve initial facts and disposition. Selected passages retain source order. This scans every sentence but **does not neurally encode every source token**; retention and target clipping are reported in the audit.

The abstracter uses the original RMSNorm/RoPE/SwiGLU/tied-embedding backbone. Encoder–decoder uses bidirectional source attention plus causal target attention and cross-attention. Other families use response-only causal loss. The local-GQA variant keeps its original GQA weights but expands its attention window to global attention during adaptation, allowing summaries to use the selected source throughout generation. The hybrid preserves the original LSTM→Transformer computation and skips source padding in the recurrent state.

| Run | Original family | Adaptation | Role |
|---|---|---|---|
| 01 | Encoder–decoder | Full adaptation | Main score candidate |
| 02 | Decoder | LoRMA+ rank 16 | Required multiplicative adaptation |
| 03 | Local GQA | Two bottleneck adapters per block | Parameter-efficient comparison |
| 04 | LSTM + Transformer | LoRA rank 16 | Hybrid-family comparison |
| 05 | Decoder | 32 learned input prompt embeddings | Very small trainable parameter set |
| 06 | Smaller decoder | Sequence-level KD + gold supervision | Genuine smaller student |

The student starts from uniformly selected layers of the original Phase-1 decoder, rather than a task-adapted parent. The strongest completed teacher generates **training-only** targets once. A deterministic 50:50 teacher/gold target choice approximates the mixture NLL on those cases. It uses greedy sequence KD as a speed-oriented approximation, not token-logit KL distillation.

Research applied (links and precise limitations):

- [Shukla et al., AACL 2022, §6.1–6.2](https://aclanthology.org/2022.aacl-main.77/): salient-context extraction followed by abstraction addresses long legal inputs. Here the selector is a lightweight train-only lexical regressor and the abstracter is your own foundation; their pretrained BERT/BART models are **not** imported.
- [Bihany et al., ACL Findings 2025, Eq. 8](https://aclanthology.org/2025.findings-acl.527/): implement LoRMA+ as $y=W_0x+\frac{\alpha}{r}BA(W_0x)$, with $B=0$ initially. This is distinct from LoRA, $W_0x+\frac{\alpha}{r}BAx$.
- [Hu et al., LoRA](https://arxiv.org/abs/2106.09685): train low-rank additive updates. Both LoRA and LoRMA+ are temporarily merged into frozen matrices for decoding, then original trainable modules are restored.
- [Houlsby et al., ICML 2019](https://proceedings.mlr.press/v97/houlsby19a.html): near-identity bottleneck adapters after attention and feed-forward residuals. Our normalized GELU adapters are a Houlsby-style variant, not an exact BERT reproduction.
- [Lester et al., EMNLP 2021](https://aclanthology.org/2021.emnlp-main.243/): learn continuous input prompts while freezing the generator. Their strong scale-dependent results do not establish that this will match full adaptation on a 50M model; this run is a comparison.
- [Kim & Rush, EMNLP 2016, §3.2](https://aclanthology.org/D16-1139/): cache teacher-generated sequences and train a smaller student with NLL. Their reported translation speedups are not forecasts for this task.
- [SummN, ACL 2022, §3](https://aclanthology.org/2022.acl-long.112/): evaluated as an alternative. It trains separate coarse/fine summarizers and adds generation stages; the default here uses extraction→abstraction to avoid that cost. This notebook **does not claim to implement SummN**.

Training cost reductions: cached source selection/tokenization; two-rank DDP with `no_sync()` during gradient accumulation; FP16 + GradScaler; checkpointed Transformer blocks; vocabulary projection only on summary positions and in small chunks; KV-cached batched decoding; merged low-rank inference weights; held-out early stopping; epoch-boundary resume; reuse of completed submissions. T4 does not support BF16 or FlashAttention-2 kernels. SDPA is used with portable fallback; the local mask is not claimed to give sparse-kernel speedups.

## Runtime modules

These cells write the complete implementation into `PROJECT/code`. The original Phase-1 class definitions are embedded unchanged. The summarization wrapper adds masking, generation, PEFT and loss without changing the checkpoint parameter names before restoration.

In [ ]:
# evaluate_phase6_official.py
module_source = r'''#!/usr/bin/env python3
"""Local evaluator for Phase 6.

Requires: rouge-score, bert-score, torch.
Truth JSON: {"doc_id":"reference summary", ...}
Prediction JSON: either {"doc_id":"summary", ...} or [{"id":"...","summary":"..."}, ...]
"""
import argparse,json,sys
try:
    from rouge_score import tokenizers, rouge_scorer
    import bert_score
except ImportError as e:
    print('Missing dependency. Install requirements.txt first.', file=sys.stderr); raise

def load(path):
    with open(path,encoding='utf-8') as f: return json.load(f)

def lcs(a,b):
    if len(a)>len(b): a,b=b,a
    masks={}
    for i,t in enumerate(a): masks[t]=masks.get(t,0)|(1<<i)
    s=0
    for t in b:
        m=masks.get(t,0); x=m|s; y=(s<<1)|1; s=x & ~(x-y)
    return s.bit_count() if hasattr(int,'bit_count') else bin(s).count('1')

def read_pred(x):
    if isinstance(x,dict): out=x
    elif isinstance(x,list):
        out={}
        for r in x:
            if not isinstance(r,dict) or 'id' not in r or 'summary' not in r: raise ValueError('Each prediction item must contain id and summary.')
            if r['id'] in out: raise ValueError(f'Duplicate prediction ID: {r["id"]}')
            out[r['id']]=r['summary']
    else: raise ValueError('Prediction must be a JSON object or list of objects.')
    for k,v in out.items():
        if not isinstance(k,str) or not isinstance(v,str) or not v.strip(): raise ValueError(f'Invalid summary for ID {k!r}.')
    return out

def main():
    ap=argparse.ArgumentParser(description='Evaluate Phase 6 summaries on your own held-out split.')
    ap.add_argument('--truth',required=True); ap.add_argument('--prediction',required=True); a=ap.parse_args()
    t=load(a.truth); p=read_pred(load(a.prediction))
    if not isinstance(t,dict) or not t: raise ValueError('Truth must be a non-empty JSON object.')
    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")
    preds=[p[k].strip() for k in t]; refs=[t[k].strip() for k in t]
    scorer=rouge_scorer.RougeScorer(['rouge1'],use_stemmer=True); tok=tokenizers.DefaultTokenizer(use_stemmer=True)
    r1=rl=0.0
    for pr,rf in zip(preds,refs):
        s=scorer.score(rf,pr); r1+=s['rouge1'].fmeasure
        pt=tok.tokenize(pr); rt=tok.tokenize(rf)
        if not pt and not rt: x=1.0
        elif not pt or not rt: x=0.0
        else:
            z=lcs(pt,rt); pp=z/len(pt); rr=z/len(rt); x=0.0 if pp+rr==0 else 2*pp*rr/(pp+rr)
        rl+=x
    n=len(t); r1/=n; rl/=n
    try:
        import torch; device='cuda' if torch.cuda.is_available() else 'cpu'
    except ImportError: device=None
    _,_,f=bert_score.score(preds,refs,lang='en',batch_size=8,verbose=False,**({'device':device} if device else {}))
    print(f'Documents: {n}'); print(f'BERTScore F1: {float(f.mean().item()):.6f}'); print(f'ROUGE-1: {r1:.6f}'); print(f'ROUGE-L: {rl:.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e: print(f'EVALUATION FAILED: {e}',file=sys.stderr); sys.exit(1)
'''
(CODE / 'evaluate_phase6_official.py').write_text(module_source, encoding='utf-8')
print('Wrote evaluate_phase6_official.py')

In [ ]:
# legal_models.py
module_source = r'''"""Four randomly initialized legal language models; no downloaded model weights."""
import math
from dataclasses import dataclass, asdict
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint


@dataclass
class ModelConfig:
    architecture: str = "decoder"
    vocab_size: int = 32768
    dim: int = 512
    heads: int = 8
    kv_heads: int = 8
    layers: int = 8
    ff_dim: int = 2048
    encoder_layers: int = 0
    rnn_layers: int = 0
    local_window: int = 0
    max_positions: int = 4096
    dropout: float = 0.0
    gradient_checkpointing: bool = False


def architecture_config(name, **overrides):
    c = ModelConfig(architecture=name)
    if name == "decoder":
        pass
    elif name == "local_gqa":
        c.kv_heads, c.layers, c.local_window = 2, 9, 256
    elif name == "hybrid":
        c.layers, c.rnn_layers = 7, 2
    elif name == "encoder_decoder":
        c.layers, c.encoder_layers = 4, 3
    else:
        raise ValueError(f"Unknown architecture: {name}")
    for k, v in overrides.items():
        if not hasattr(c, k):
            raise ValueError(k)
        setattr(c, k, v)
    assert c.dim % c.heads == 0 and c.heads % c.kv_heads == 0
    assert (c.dim // c.heads) % 2 == 0
    return c


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        # Reduction in FP32 is useful with T4 FP16 training.
        y = x.float() * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return y.to(x.dtype) * self.weight.to(x.dtype)


class Rotary(nn.Module):
    def __init__(self, head_dim, max_positions):
        super().__init__()
        inv = 1.0 / (10000 ** (torch.arange(0, head_dim, 2).float() / head_dim))
        angles = torch.outer(torch.arange(max_positions).float(), inv)
        self.register_buffer("cos", angles.cos(), persistent=False)
        self.register_buffer("sin", angles.sin(), persistent=False)

    def forward(self, x, offset=0):
        n = x.size(-2)
        if offset + n > self.cos.size(0):
            raise ValueError("Position limit exceeded; increase max_positions in the model config.")
        cos = self.cos[offset:offset + n].to(x.dtype)[None, None]
        sin = self.sin[offset:offset + n].to(x.dtype)[None, None]
        a, b = x[..., 0::2], x[..., 1::2]
        return torch.stack((a * cos - b * sin, a * sin + b * cos), dim=-1).flatten(-2)


class Attention(nn.Module):
    def __init__(self, c, causal=True, cross=False):
        super().__init__()
        self.heads, self.kv_heads = c.heads, c.kv_heads
        self.hd = c.dim // c.heads
        self.q = nn.Linear(c.dim, c.dim, bias=False)
        self.k = nn.Linear(c.dim, self.kv_heads * self.hd, bias=False)
        self.v = nn.Linear(c.dim, self.kv_heads * self.hd, bias=False)
        self.out = nn.Linear(c.dim, c.dim, bias=False)
        self.rotary = None if cross else Rotary(self.hd, c.max_positions)
        self.causal, self.cross = causal, cross
        self.window = c.local_window if causal and not cross else 0
        self.dropout = c.dropout

    def forward(self, x, memory=None, past=None, position=0, use_cache=False):
        b, t, _ = x.shape
        q = self.q(x).view(b, t, self.heads, self.hd).transpose(1, 2)
        if self.cross and past is not None:
            k, v = past  # Encoder K/V projections are also cached during decoding.
        else:
            source = x if memory is None else memory
            s = source.size(1)
            k = self.k(source).view(b, s, self.kv_heads, self.hd).transpose(1, 2)
            v = self.v(source).view(b, s, self.kv_heads, self.hd).transpose(1, 2)
            if self.rotary is not None:
                k = self.rotary(k, position)
            if past is not None:
                k, v = torch.cat((past[0], k), -2), torch.cat((past[1], v), -2)
        if self.rotary is not None:
            q = self.rotary(q, position)
        new_cache = (k, v) if use_cache else None
        # Portable GQA: project/cache fewer KV heads, repeat only for the SDPA call.
        # This is not a block-sparse implementation or a guaranteed training speedup.
        if self.kv_heads != self.heads:
            repeats = self.heads // self.kv_heads
            k = k.repeat_interleave(repeats, dim=1)
            v = v.repeat_interleave(repeats, dim=1)
        mask, is_causal = None, False
        if self.causal:
            s = k.size(-2)
            if self.window or (past is not None and t > 1):
                qi = torch.arange(s - t, s, device=x.device)[:, None]
                ki = torch.arange(s, device=x.device)[None, :]
                mask = ki <= qi
                if self.window:
                    mask = mask & ((qi - ki) < self.window)
            elif past is None:
                is_causal = True
            # Cached single-token query must see all prior keys. is_causal=True
            # here would incorrectly use the upper-left non-square mask.
        y = F.scaled_dot_product_attention(
            q, k, v, attn_mask=mask, is_causal=is_causal,
            dropout_p=self.dropout if self.training else 0.0,
        )
        y = self.out(y.transpose(1, 2).contiguous().view(b, t, -1))
        return y, new_cache


class SwiGLU(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.gate = nn.Linear(c.dim, c.ff_dim, bias=False)
        self.up = nn.Linear(c.dim, c.ff_dim, bias=False)
        self.down = nn.Linear(c.ff_dim, c.dim, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class Block(nn.Module):
    def __init__(self, c, encoder=False):
        super().__init__()
        self.attn_norm, self.ff_norm = RMSNorm(c.dim), RMSNorm(c.dim)
        self.attn = Attention(c, causal=not encoder)
        self.ff = SwiGLU(c)
        self.drop = nn.Dropout(c.dropout)

    def forward(self, x, past=None, position=0, use_cache=False):
        y, cache = self.attn(self.attn_norm(x), past=past, position=position, use_cache=use_cache)
        x = x + self.drop(y)
        x = x + self.drop(self.ff(self.ff_norm(x)))
        return (x, cache) if use_cache else x


class CrossBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.self_norm, self.cross_norm, self.ff_norm = (RMSNorm(c.dim) for _ in range(3))
        self.self_attn = Attention(c)
        self.cross_attn = Attention(c, causal=False, cross=True)
        self.ff, self.drop = SwiGLU(c), nn.Dropout(c.dropout)

    def forward(self, x, memory, past=None, cross_past=None, position=0, use_cache=False):
        y, cache = self.self_attn(self.self_norm(x), past=past, position=position, use_cache=use_cache)
        x = x + self.drop(y)
        y, cross_cache = self.cross_attn(self.cross_norm(x), memory=memory, past=cross_past, use_cache=use_cache)
        x = x + self.drop(y)
        x = x + self.drop(self.ff(self.ff_norm(x)))
        return (x, cache, cross_cache) if use_cache else x


class LegalLM(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.config = c
        self.embedding = nn.Embedding(c.vocab_size, c.dim)
        # Output projection uses embedding.weight; there is no second vocabulary matrix.
        if c.architecture == "encoder_decoder":
            self.encoder = nn.ModuleList(Block(c, encoder=True) for _ in range(c.encoder_layers))
            self.encoder_norm = RMSNorm(c.dim)
            self.blocks = nn.ModuleList(CrossBlock(c) for _ in range(c.layers))
        else:
            self.blocks = nn.ModuleList(Block(c) for _ in range(c.layers))
        if c.rnn_layers:
            self.rnn = nn.LSTM(c.dim, c.dim, num_layers=c.rnn_layers, batch_first=True)
            self.rnn_norm = RMSNorm(c.dim)
        self.final_norm = RMSNorm(c.dim)
        self.apply(self._init)
        # Scale residual projections to keep initial residual variance controlled.
        std = 0.02 / math.sqrt(2 * c.layers)
        for name, p in self.named_parameters():
            if name.endswith(("attn.out.weight", "self_attn.out.weight", "cross_attn.out.weight", "ff.down.weight")):
                nn.init.normal_(p, mean=0.0, std=std)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        elif isinstance(m, nn.LSTM):
            for name, p in m.named_parameters():
                if "weight" in name:
                    nn.init.normal_(p, mean=0.0, std=0.02)
                else:
                    nn.init.zeros_(p)

    def encode(self, source_ids):
        x = self.embedding(source_ids)
        for block in self.encoder:
            if self.config.gradient_checkpointing and self.training:
                x = checkpoint(block, x, use_reentrant=False)
            else:
                x = block(x)
        return self.encoder_norm(x)

    def forward(self, input_ids, source_ids=None, state=None, use_cache=False):
        state = state or {}
        position = state.get("position", 0)
        x = self.embedding(input_ids)
        new_state = {"position": position + input_ids.size(1), "self": []}
        if self.config.rnn_layers:
            y, recurrent_state = self.rnn(x, state.get("rnn"))
            x = x + self.rnn_norm(y)
            new_state["rnn"] = recurrent_state
        if self.config.architecture == "encoder_decoder":
            memory = state.get("memory")
            if memory is None:
                if source_ids is None:
                    raise ValueError("encoder_decoder requires source_ids on the first call.")
                memory = self.encode(source_ids)
            new_state.update(memory=memory, cross=[])
            for i, block in enumerate(self.blocks):
                if use_cache:
                    p = state.get("self", [None] * len(self.blocks))[i]
                    cp = state.get("cross", [None] * len(self.blocks))[i]
                    x, cache, cc = block(x, memory, p, cp, position, True)
                    new_state["self"].append(cache)
                    new_state["cross"].append(cc)
                elif self.config.gradient_checkpointing and self.training:
                    x = checkpoint(block, x, memory, use_reentrant=False)
                else:
                    x = block(x, memory)
        else:
            for i, block in enumerate(self.blocks):
                if use_cache:
                    p = state.get("self", [None] * len(self.blocks))[i]
                    x, cache = block(x, p, position, True)
                    new_state["self"].append(cache)
                elif self.config.gradient_checkpointing and self.training:
                    x = checkpoint(block, x, use_reentrant=False)
                else:
                    x = block(x)
        x = self.final_norm(x)
        return (x, new_state) if use_cache else x

    def project(self, hidden):
        return F.linear(hidden, self.embedding.weight)


def parameter_count(model):
    # PyTorch de-duplicates shared Parameter objects here.
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def parameter_formula(c):
    d, h, kv = c.dim, c.dim // c.heads, c.kv_heads
    attention = 2 * d * d + 2 * d * kv * h
    ff = 3 * d * c.ff_dim
    total = c.vocab_size * d + d
    if c.architecture == "encoder_decoder":
        total += c.encoder_layers * (attention + ff + 2 * d) + d
        total += c.layers * (2 * attention + ff + 3 * d)
    else:
        total += c.layers * (attention + ff + 2 * d)
    if c.rnn_layers:
        total += c.rnn_layers * (8 * d * d + 8 * d) + d
    return total


def make_batch(blocks, architecture, prefix_fraction=0.80, objective="full"):
    # blocks has T+1 tokens. The target suffix is identical across architectures.
    t = blocks.size(1) - 1
    split = max(1, min(t, int(t * prefix_fraction)))
    if architecture == "encoder_decoder":
        source = blocks[:, :split]
        inputs, targets = blocks[:, split - 1:-1], blocks[:, split:]
    else:
        source = None
        inputs, targets = blocks[:, :-1], blocks[:, 1:].clone()
        if objective == "continuation":
            targets[:, :split - 1] = -100
    return inputs, targets, source


def language_model_loss(hidden, targets, weight, chunk_tokens=256, checkpoint_chunks=False):
    """Sum/token count lets validation aggregate the true token-weighted NLL."""
    hidden, targets = hidden.reshape(-1, hidden.size(-1)), targets.reshape(-1)
    total = hidden.new_zeros((), dtype=torch.float32)
    count = (targets != -100).sum()
    def loss_chunk(h, y, w):
        # Keep logits in autocast precision until the bounded FP32 CE conversion.
        return F.cross_entropy(F.linear(h, w).float(), y, ignore_index=-100, reduction="sum")
    for i in range(0, targets.numel(), chunk_tokens):
        h, y = hidden[i:i + chunk_tokens], targets[i:i + chunk_tokens]
        if checkpoint_chunks and torch.is_grad_enabled():
            total = total + checkpoint(loss_chunk, h, y, weight, use_reentrant=False)
        else:
            total = total + loss_chunk(h, y, weight)
    return total / count.clamp_min(1), total.detach(), count.detach()


if __name__ == "__main__":
    for name in ("decoder", "encoder_decoder", "local_gqa", "hybrid"):
        c = architecture_config(name)
        with torch.device("meta"):
            model = LegalLM(c)
        n = parameter_count(model)
        assert n == parameter_formula(c) and 50_000_000 <= n <= 300_000_000
        print(name, f"{n:,}", asdict(c))
'''
(CODE / 'legal_models.py').write_text(module_source, encoding='utf-8')
print('Wrote legal_models.py')

In [ ]:
# s6_data.py
module_source = r'''"""Official Phase-6 data, train-only salience selector, auditable cached contexts."""
import hashlib,json,re,os,pickle,collections
from pathlib import Path
import numpy as np
from tokenizers import Tokenizer

FILES=['Train/train.json','Test/test.json']
def digest(x):return hashlib.sha256(json.dumps(x,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
def file_hash(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''):h.update(b)
    return h.hexdigest()
def read_json(p):
    def unique(pairs):
        d={}
        for k,v in pairs:
            if k in d:raise ValueError(f'Duplicate JSON mapping key: {k}')
            d[k]=v
        return d
    return json.loads(Path(p).read_text(encoding='utf-8-sig'),object_pairs_hook=unique)
def atomic_json(p,x):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)
    t=p.with_name(p.name+'.tmp');t.write_text(json.dumps(x,ensure_ascii=False,indent=2));t.replace(p)
def alias(r,names):
    keys=[k for k in names if k in r]
    if len(keys)>1 and any(r[k]!=r[keys[0]] for k in keys):raise ValueError(f'Conflicting fields: {keys}')
    return r[keys[0]] if keys else None
DOCUMENT_FIELDS=('document','text','sentences')
SUMMARY_FIELDS=('summary','summaries','abstract','reference_summary','reference_summaries','gold_summary','summary_text','abstractive_summary','reference','highlights','target')
def text_value(v,fields,depth=0):
    """Normalize text and explicit text containers, never stringify arbitrary metadata."""
    if depth>5:return None
    if isinstance(v,str):return v.strip() or None
    if isinstance(v,list):
        if all(isinstance(s,str) or s is None for s in v):
            return '\n'.join(s.strip() for s in v if isinstance(s,str) and s.strip()) or None
        # A list of structured references can contain alternatives; use the first usable one.
        for item in v:
            result=text_value(item,fields,depth+1)
            if result:return result
    elif isinstance(v,dict):
        for key in fields:
            if key in v:
                result=text_value(v[key],fields,depth+1)
                if result:return result
    return None
def records(v,split,audit=None):
    if isinstance(v,dict) and len(v)==1 and next(iter(v)) in ('data','records','documents'):v=next(iter(v.values()))
    if isinstance(v,list):items=[(None,r) for r in v]
    elif isinstance(v,dict):
        ids=alias(v,('id','ids','doc_id','case_id'));texts=alias(v,DOCUMENT_FIELDS)
        summary_columns={} if split=='test' else {k:v[k] for k in SUMMARY_FIELDS if k in v}
        if isinstance(ids,str) and texts is not None:items=[(None,v)]
        elif isinstance(ids,list) and isinstance(texts,list):
            if len(ids)!=len(texts) or any(v is not None and (not isinstance(v,list) or len(v)!=len(ids)) for v in summary_columns.values()):raise ValueError(f'{split}: column lengths differ; cannot align documents and summaries by position.')
            items=[(None,{'id':k,'document':texts[i],**{f:col[i] for f,col in summary_columns.items() if col is not None}}) for i,k in enumerate(ids)]
        elif isinstance(texts,dict) and ids is None:
            if any(col is not None and (not isinstance(col,dict) or set(col)-set(texts)) for col in summary_columns.values()):raise ValueError(f'{split}: summary columns contain unexpected IDs or are not mappings.')
            items=[(None,{'id':k,'document':t,**{f:col.get(k) for f,col in summary_columns.items() if col is not None}}) for k,t in texts.items()]
        else:items=list(v.items())
    else:raise ValueError('Expected list of records or ID mapping.')
    out={};duplicates=0;skipped=[];fields=collections.Counter()
    for key,r in items:
        if isinstance(r,(str,list)) and key is not None:r={'id':key,'document':r}
        if not isinstance(r,dict):raise ValueError(f'{split}: record must be an object; found {type(r).__name__}.')
        did=alias(r,('id','doc_id','case_id'));did=key if did is None else did
        if not isinstance(did,str) or not did.strip():raise ValueError('Original nonempty string IDs required; IDs are never invented.')
        if key is not None and key!=did:raise ValueError('Mapping key and explicit ID differ.')
        text=text_value(alias(r,DOCUMENT_FIELDS),DOCUMENT_FIELDS+('content','sentence'))
        # Deliberately do not even inspect supplied test reference fields.
        summary=None
        if split!='test':
            for field in SUMMARY_FIELDS:
                if field not in r:continue
                summary=text_value(r[field],SUMMARY_FIELDS+('text','content','sentences','sentence'))
                if summary:
                    fields[field]+=1;break
        if not text or (split!='test' and not summary):
            reason='missing or unusable document' if not text else 'missing or unusable reference summary'
            if split=='test':raise ValueError(f'test ID {did!r}: {reason}; cannot silently drop an official test ID.')
            skipped.append({'id':did,'reason':reason,'field_types':{k:type(val).__name__ for k,val in r.items()}})
            continue
        row={'id':did,'document':text,'summary':summary,'split':split}
        if did in out:
            if row!=out[did]:raise ValueError(f'{split}: conflicting duplicate ID {did}; inspect source instead of inventing an ID.')
            duplicates+=1;continue
        out[did]=row
    if audit is not None:audit.update(input_rows=len(items),usable_rows=len(out),skipped_rows=skipped,reference_fields_used=dict(fields))
    if not out:raise ValueError(f'No usable {split} records. Examples (IDs and field types only): {skipped[:5]}')
    return list(out.values()),duplicates
def normhash(text):return digest(' '.join(text.lower().split()))
def split_data(root,seed=781,dev_fraction=.08,dev_max=512):
    train_audit={};test_audit={}
    train,duptr=records(read_json(Path(root)/FILES[0]),'train',train_audit);test,dupte=records(read_json(Path(root)/FILES[1]),'test',test_audit)
    forbidden_ids={r['id'] for r in test};forbidden_text={normhash(r['document']) for r in test}
    removed=[r['id'] for r in train if r['id'] in forbidden_ids or normhash(r['document']) in forbidden_text]
    removed_ids=set(removed);train=[r for r in train if r['id'] not in removed_ids]
    # Group exact normalized document copies before splitting; retain one training reference.
    groups=collections.defaultdict(list)
    for r in train:groups[normhash(r['document'])].append(r)
    keys=sorted(groups);np.random.default_rng(seed).shuffle(keys)
    if len(keys)<4:raise ValueError(f'Need at least four distinct labeled train documents after cleanup; got {len(keys)}. Skipped rows (examples): {train_audit["skipped_rows"][:5]}')
    n=max(1,min(dev_max,round(len(keys)*dev_fraction),len(keys)-2));held=set(keys[:n])
    rows=[];conflicting=0
    for key in keys:
        group=groups[key];conflicting+=len({r['summary'] for r in group})>1
        r=dict(sorted(group,key=lambda x:x['id'])[0]);r['split']='dev' if key in held else 'train';rows.append(r)
    rows+=test
    audit={'duplicate_id_copies_removed':{'train':duptr,'test':dupte},'train_test_overlap_removed':removed,
           'train_records':train_audit,'test_records':test_audit,
           'normalized_document_copies_collapsed':sum(len(v)-1 for v in groups.values()),
           'copy_groups_with_multiple_references':conflicting,'dev_group_count':n,
           'test_references':'discarded without inspection; never cached','seed':seed}
    return rows,audit
def sentences(text):
    # Deterministic, no NLTK resource downloads. Keep sentence boundaries and source order.
    parts=[s.strip() for s in re.split(r'(?<=[.!?])\s+(?=[A-Z0-9(])|\n+',text) if s.strip()]
    # Cap individual fragments for bounded selector features; no head-only document truncation.
    return [s[i:i+1800] for s in parts for i in range(0,len(s),1800)] or [text]
def wordset(s):return set(re.findall(r'[a-z0-9]+',s.lower()))
CUE=re.compile(r'\b(held|hold|therefore|dismissed|allowed|appeal|issue|court|section|petitioner|respondent|conclusion|ordered|judgment)\b',re.I)
class Selector:
    """Hashing salience regression + position/cue features; fitted on training references only."""
    def __init__(self,seed=781):
        from sklearn.feature_extraction.text import HashingVectorizer
        from sklearn.linear_model import SGDRegressor
        self.vectorizer=HashingVectorizer(n_features=2**18,ngram_range=(1,2),alternate_sign=False,norm='l2')
        self.model=SGDRegressor(loss='huber',epsilon=.15,alpha=1e-5,learning_rate='invscaling',eta0=.05,random_state=seed)
    def features(self,ss):
        from scipy.sparse import hstack,csr_matrix
        n=len(ss);extra=np.asarray([[i/max(1,n-1),float(i<3),float(i>=n-3),min(5,len(CUE.findall(s)))/5,min(1,len(s)/600)] for i,s in enumerate(ss)],dtype=np.float32)
        return hstack([self.vectorizer.transform(ss),csr_matrix(extra)],format='csr')
    def fit(self,rows,passes=2):
        for _ in range(passes):
            for r in rows:
                ss=sentences(r['document']);ref=wordset(r['summary']);y=[]
                for s in ss:
                    w=wordset(s);overlap=len(w&ref)
                    # Sentence-level lexical F score is a cheap salience training signal, not the evaluation metric.
                    y.append(2*overlap/max(1,len(w)+min(len(ref),80)))
                self.model.partial_fit(self.features(ss),np.asarray(y,dtype=np.float32))
        return self
    def select(self,text,tok,budget):
        ss=sentences(text);encoded=tok.encode_batch(ss,add_special_tokens=False);ids=[e.ids for e in encoded]
        if sum(map(len,ids))+len(ss)<=budget:return tok.encode('\n'.join(ss),add_special_tokens=False).ids[:budget],len(ss),len(ss)
        scores=self.model.predict(self.features(ss));sets=list(map(wordset,ss));selected=[];used=0;remaining=set(range(len(ss)))
        # Reserve a little space for facts and final disposition; fill most of the context by learned salience + MMR.
        for i in list(dict.fromkeys([0,len(ss)-1])):
            take=min(len(ids[i]),max(1,budget//12))
            if take:selected.append((i,ids[i][:take]));used+=take+1;remaining.discard(i)
        while remaining and used<budget-8:
            def score(i):
                redundancy=max((len(sets[i]&sets[j])/max(1,len(sets[i]|sets[j])) for j,_ in selected),default=0)
                return float(scores[i])-.15*redundancy
            i=max(remaining,key=lambda j:(score(j),-j));remaining.remove(i)
            room=budget-used-1
            if len(ids[i])>room and selected:continue
            selected.append((i,ids[i][:room]));used+=min(room,len(ids[i]))+1
        sep=tok.encode('\n',add_special_tokens=False).ids
        result=[]
        for _,tokens in sorted(selected):result+=tokens+sep
        return result[:budget],len(selected),len(ss)
def special_ids(tok,overrides=None):
    overrides=overrides or {};out={}
    for name,candidates in {'pad':['<pad>','[PAD]'],'bos':['<bos>','[BOS]','<s>'],'eos':['<eos>','[EOS]','</s>']}.items():
        value=overrides.get(name)
        if value is None:value=next((tok.token_to_id(s) for s in candidates if tok.token_to_id(s) is not None),None)
        if value is None:raise ValueError(f'Cannot identify {name} token. Set SPECIAL_IDS explicitly from tokenizer vocabulary.')
        if not 0<=int(value)<tok.get_vocab_size():raise ValueError('Special ID outside tokenizer vocabulary.')
        out[name]=int(value)
    if len(set(out.values()))!=3:raise ValueError('PAD, BOS and EOS must be distinct existing tokens.')
    return out
def prepare_data(root,tokenizer_path,output,settings):
    import torch
    output=Path(output);output.mkdir(parents=True,exist_ok=True);path=output/'cache_v4.pt'
    identity=digest({'files':{n:file_hash(Path(root)/n) for n in FILES},'tokenizer':file_hash(tokenizer_path),'settings':settings,'version':4})
    if path.exists():
        cache=torch.load(path,map_location='cpu',weights_only=False)
        if cache['fingerprint']!=identity:raise ValueError('Inputs/settings changed. Choose a new PROJECT to rebuild cleanly.')
        if any(r['target'] is not None or r['summary'] is not None for r in cache['rows'] if r['split']=='test'):raise ValueError('Unsafe cache has test references.')
        return str(path)
    rows,audit=split_data(root,settings['seed'],settings['dev_fraction'],settings['dev_max'])
    skipped=audit['train_records']['skipped_rows']
    print(f'Training input audit: {audit["train_records"]["usable_rows"]} labeled records; {len(skipped)} unusable rows excluded; reference fields: {audit["train_records"]["reference_fields_used"]}',flush=True)
    if skipped:print(f'Excluded training IDs (first 10): {[r["id"] for r in skipped[:10]]}; full details in token_cache/audit.json.',flush=True)
    atomic_json(output/'audit.json',audit)
    tok=Tokenizer.from_file(str(tokenizer_path));tok.no_padding();tok.no_truncation();special=special_ids(tok,settings.get('special_ids'))
    train=[r for r in rows if r['split']=='train'];selector=Selector(settings['seed']).fit(train)
    tmp=output/'selector.pkl.tmp'
    with tmp.open('wb') as f:pickle.dump(selector,f)
    tmp.replace(output/'selector.pkl')
    target_lengths=[len(e.ids) for e in tok.encode_batch([r['summary'] for r in train],add_special_tokens=False)]
    target_cap=min(settings['target_tokens'],max(64,int(np.quantile(target_lengths,.95))+1))
    retained=[];clipped=0;cached=[]
    for i,r in enumerate(rows):
        source,nsel,ns=selector.select(r['document'],tok,settings['source_tokens']-64)
        prompt=tok.encode('Summarize the legal judgment, including the material facts, legal issue, reasoning and final decision.\nJudgment:\n',add_special_tokens=False).ids+source+tok.encode('\nSummary:\n',add_special_tokens=False).ids
        prompt=prompt[:settings['source_tokens']]
        target=None
        if r['split']!='test':
            target=tok.encode(r['summary'],add_special_tokens=False).ids;clipped+=len(target)>=target_cap
            target=target[:target_cap-1]+[special['eos']]
        cached.append({'id':r['id'],'split':r['split'],'source':prompt,'target':target,'summary':r['summary'],
                       'document_sha256':normhash(r['document']),'selected_sentences':nsel,'sentences':ns})
        retained.append(nsel/ns)
        if (i+1)%500==0:print('Cached documents:',i+1,flush=True)
    audit.update(counts={s:sum(r['split']==s for r in cached) for s in ('train','dev','test')},
                 target_token_quantiles_train={str(q):float(np.quantile(target_lengths,q)) for q in (.5,.9,.95,.99)},
                 target_cap=target_cap,references_clipped_at_target_cap=clipped,mean_sentence_retention=float(np.mean(retained)),
                 source_policy='Learned train-only sentence salience + MMR + first/last quota; neural input is a selected subset, not every document token.')
    value={'rows':cached,'fingerprint':identity,'special':special,'tokenizer_path':str(tokenizer_path),
           'generation_blocked_ids':[r['id'] for r in json.loads(tok.to_str()).get('added_tokens',[]) if r.get('special') and r['id']!=special['eos']],
           'tokenizer_sha256':file_hash(tokenizer_path),'source_tokens':settings['source_tokens'],'target_tokens':target_cap,'audit':audit}
    tmp=path.with_name(path.name+'.tmp');torch.save(value,tmp);tmp.replace(path);atomic_json(output/'audit.json',audit)
    return str(path)
'''
(CODE/'s6_data.py').write_text(module_source, encoding='utf-8')
print('Wrote s6_data.py')

In [ ]:
# s6_metrics.py
module_source = r'''"""Official-equivalent metrics; pretrained BERT is used for detached text evaluation only."""
import torch
_scorer=None
def release():
    global _scorer
    import gc
    _scorer=None;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
def score(predictions,references,device='cpu'):
    global _scorer
    from rouge_score import rouge_scorer,tokenizers
    from bert_score import BERTScorer
    from evaluate_phase6_official import lcs
    if len(predictions)!=len(references) or not predictions:raise ValueError('Metric input lengths differ/empty.')
    if _scorer is None or str(_scorer.device)!=str(device):
        _scorer=BERTScorer(lang='en',batch_size=8,device=str(device),rescale_with_baseline=False,idf=False)
    # The provided official evaluator uses bert_score.score(..., lang='en'), without rescaling or IDF.
    with torch.inference_mode():_,_,f=_scorer.score(predictions,references,batch_size=8)
    rs=rouge_scorer.RougeScorer(['rouge1'],use_stemmer=True);tok=tokenizers.DefaultTokenizer(use_stemmer=True)
    r1=[];rl=[]
    for p,r in zip(predictions,references):
        r1.append(rs.score(r,p)['rouge1'].fmeasure);pp=tok.tokenize(p);rr=tok.tokenize(r)
        rl.append(2*lcs(pp,rr)/max(1,len(pp)+len(rr)) if pp or rr else 1.)
    return {'bertscore_f1':float(f.mean()),'rouge1_f1':sum(r1)/len(r1),'rougeL_f1':sum(rl)/len(rl),
            'documents':len(predictions),'bertscore_hash':_scorer.hash,'metric_model':'evaluation only; never part of generator/optimizer'}
'''
(CODE / 's6_metrics.py').write_text(module_source, encoding='utf-8')
print('Wrote s6_metrics.py')

In [ ]:
# s6_model.py
module_source = r'''"""Original Phase-1 weights with conditional generation, PEFT and KV caches."""
import math,types,contextlib
from dataclasses import fields
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from legal_models import LegalLM,ModelConfig,Attention
from s6_data import file_hash

class LowRank(nn.Module):
    def __init__(self,base,rank,alpha,multiplicative=False):
        super().__init__();self.base=base;self.scale=alpha/rank;self.multiplicative=multiplicative;base.requires_grad_(False)
        self.A=nn.Linear(base.out_features if multiplicative else base.in_features,rank,bias=False)
        self.B=nn.Linear(rank,base.out_features,bias=False)
        nn.init.normal_(self.A.weight,std=.02);nn.init.zeros_(self.B.weight)
    def forward(self,x):
        y=self.base(x)
        # LoRMA+ Eq.8: (I + alpha/r BA) W x. LoRA: W x + alpha/r BA x.
        return y+self.scale*self.B(self.A(y if self.multiplicative else x))
@contextlib.contextmanager
def merged_adapters(model):
    """Exact FP32 weight algebra for LoRA/LoRMA+: remove adapter matmul overhead while decoding."""
    replacements=[]
    with torch.no_grad():
        for name,m in list(model.named_modules()):
            if not isinstance(m,LowRank):continue
            parent_name,attr=name.rsplit('.',1);parent=model.get_submodule(parent_name)
            layer=nn.Linear(m.base.in_features,m.base.out_features,bias=m.base.bias is not None,device=m.base.weight.device,dtype=m.base.weight.dtype)
            update=m.B.weight@(m.A.weight@m.base.weight if m.multiplicative else m.A.weight)
            layer.weight.copy_(m.base.weight+m.scale*update)
            if m.base.bias is not None:
                bias=m.base.bias
                layer.bias.copy_(bias+m.scale*m.B.weight@(m.A.weight@bias) if m.multiplicative else bias)
            layer.requires_grad_(False);setattr(parent,attr,layer);replacements.append((parent,attr,m))
    try:yield
    finally:
        for parent,attr,original in replacements:setattr(parent,attr,original)
class Adapter(nn.Module):
    def __init__(self,d,b):
        super().__init__();self.norm=nn.LayerNorm(d);self.down=nn.Linear(d,b);self.up=nn.Linear(b,d)
        nn.init.zeros_(self.up.weight);nn.init.zeros_(self.up.bias)
    def forward(self,x):return x+self.up(F.gelu(self.down(self.norm(x))))
def attention(self,x,memory=None,past=None,position=0,use_cache=False):
    b,t,_=x.shape;q=self.q(x).view(b,t,self.heads,self.hd).transpose(1,2)
    if self.cross and past is not None:k,v=past
    else:
        src=x if memory is None else memory;s=src.size(1)
        k=self.k(src).view(b,s,self.kv_heads,self.hd).transpose(1,2);v=self.v(src).view(b,s,self.kv_heads,self.hd).transpose(1,2)
        if self.rotary is not None:k=self.rotary(k,position)
        if past is not None:k=torch.cat([past[0],k],-2);v=torch.cat([past[1],v],-2)
    if self.rotary is not None:q=self.rotary(q,position)
    cache=(k,v) if use_cache else None
    if self.heads!=self.kv_heads:k=k.repeat_interleave(self.heads//self.kv_heads,1);v=v.repeat_interleave(self.heads//self.kv_heads,1)
    s=k.shape[-2];valid=self._valid[:,:s];allowed=valid[:,None,None,:].expand(b,1,t,s)
    if self.causal:
        qi=torch.arange(position,position+t,device=x.device)[:,None];ki=torch.arange(s,device=x.device)[None,:]
        relation=ki<=qi
        if self.window:relation=relation&((qi-ki)<self.window)
        allowed=allowed&relation[None,None]
    # Every real query sees at least BOS/source; padded queries may be fully masked.
    # SDPA handles empty query rows as zero. No FP16 -1e9 masked-fill path.
    y=F.scaled_dot_product_attention(q,k,v,attn_mask=allowed,dropout_p=self.dropout if self.training else 0.)
    return self.out(y.transpose(1,2).contiguous().view(b,t,-1)),cache
def load_parent(path,tokenizer_sha,allow_tiny=False):
    saved=torch.load(path,map_location='cpu',weights_only=False)
    if saved.get('training_origin')!='random_initialization_official_phase1_only':raise ValueError('Parent must be an original own Phase-1 checkpoint, not an adapted task checkpoint.')
    if saved.get('tokenizer_sha256')!=tokenizer_sha:raise ValueError('Parent/tokenizer SHA256 mismatch.')
    c=ModelConfig(**{k:v for k,v in saved['model_config'].items() if k in {f.name for f in fields(ModelConfig)}})
    c.gradient_checkpointing=False;model=LegalLM(c);model.load_state_dict(saved['model'],strict=True)
    n=sum(p.numel() for p in model.parameters())
    if not allow_tiny and not 50_000_000<=n<=300_000_000:raise ValueError(f'Original parent has {n:,} parameters, outside 50M–300M.')
    return model
class Summarizer(nn.Module):
    def __init__(self,backbone,cfg):
        super().__init__();self.backbone=backbone;self.cfg=cfg;self.native=backbone.config.architecture=='encoder_decoder'
        method=cfg['method'];backbone.requires_grad_(method in ('full','kd'))
        if method=='kd':
            count=max(1,len(backbone.blocks)//2);indices=np.linspace(0,len(backbone.blocks)-1,count).round().astype(int).tolist()
            backbone.blocks=nn.ModuleList([backbone.blocks[i] for i in indices]);backbone.config.layers=count;self.student_layers=indices
        if method in ('lora','lorma'):
            targets={'q','k','v','out','gate','up','down'}
            replacements=[(name,m) for name,m in backbone.named_modules() if isinstance(m,nn.Linear) and name.rsplit('.',1)[-1] in targets]
            for name,m in replacements:
                parent,attr=name.rsplit('.',1);setattr(backbone.get_submodule(parent),attr,LowRank(m,cfg.get('rank',16),cfg.get('alpha',32),method=='lorma'))
        if method=='adapter':
            for block in backbone.blocks:
                block.after_attention_adapter=Adapter(backbone.config.dim,cfg.get('adapter_dim',64))
                block.after_ff_adapter=Adapter(backbone.config.dim,cfg.get('adapter_dim',64))
        if method=='prompt':
            n=cfg.get('prompt_tokens',32)
            self.prompt=nn.Parameter(backbone.embedding.weight.detach()[:n].clone())
        for m in backbone.modules():
            if isinstance(m,Attention):
                m.forward=types.MethodType(attention,m);m.dropout=cfg.get('dropout',.05)
                # Keep GQA projections; allow long-document information instead of losing source outside a 256-token window.
                if cfg.get('globalize_local',False):m.window=0
            if isinstance(m,nn.Dropout):m.p=cfg.get('dropout',.05)
        self.checkpointing=cfg.get('checkpointing',True)
    def block(self,block,x,valid,memory=None,source_valid=None,position=0,past=None,cross_past=None,cache=False):
        if self.native and memory is not None:
            block.self_attn._valid=valid;block.cross_attn._valid=source_valid
            return block(x,memory,past,cross_past,position,cache)
        block.attn._valid=valid
        if hasattr(block,'after_attention_adapter'):
            y,c=block.attn(block.attn_norm(x),past=past,position=position,use_cache=cache)
            x=block.after_attention_adapter(x+block.drop(y));x=block.after_ff_adapter(x+block.drop(block.ff(block.ff_norm(x))))
            return (x,c) if cache else x
        return block(x,past,position,cache)
    def encode(self,ids,valid):
        x=self.backbone.embedding(ids)
        for block in self.backbone.encoder:
            def f(x,valid,block=block):return self.block(block,x,valid)
            x=checkpoint(f,x,valid,use_reentrant=False) if self.training and self.checkpointing else f(x,valid)
        return self.backbone.encoder_norm(x)
    def hidden(self,ids,valid,source=None,source_valid=None,state=None,use_cache=False):
        state={} if state is None else state;pos=state.get('position',0);x=self.backbone.embedding(ids)
        if hasattr(self,'prompt') and pos==0:
            x=torch.cat([self.prompt[None].expand(len(ids),-1,-1),x],1)
            valid=torch.cat([torch.ones(len(ids),self.prompt.shape[0],device=ids.device,dtype=torch.bool),valid],1)
        if pos and 'valid' in state:valid=torch.cat([state['valid'],valid],1)
        new={'position':pos+x.size(1),'valid':valid,'self':[]}
        if hasattr(self.backbone,'rnn'):
            # Fused LSTM stays outside checkpoint recomputation (cuDNN metadata safety).
            if pos==0:
                # Skip masked source PADs in the recurrent state too, not only in attention.
                width=self.cfg['source_tokens'];lengths=valid[:,:width].sum(1).cpu()
                packed=nn.utils.rnn.pack_padded_sequence(x[:,:width],lengths,batch_first=True,enforce_sorted=False)
                py,rnn=self.backbone.rnn(packed);sy,_=nn.utils.rnn.pad_packed_sequence(py,batch_first=True,total_length=width)
                ty,rnn=self.backbone.rnn(x[:,width:],rnn);y=torch.cat([sy,ty],1)
            else:y,rnn=self.backbone.rnn(x,state.get('rnn'))
            x=x+self.backbone.rnn_norm(y);new['rnn']=rnn
        memory=None
        if self.native:
            memory=state.get('memory')
            if memory is None:memory=self.encode(source,source_valid)
            else:source_valid=state['source_valid']
            new.update(memory=memory,source_valid=source_valid,cross=[])
        for i,block in enumerate(self.backbone.blocks):
            if use_cache:
                y=self.block(block,x,valid,memory,source_valid,pos,state.get('self',[None]*len(self.backbone.blocks))[i],state.get('cross',[None]*len(self.backbone.blocks))[i] if self.native else None,True)
                x=y[0];new['self'].append(y[1])
                if self.native:new['cross'].append(y[2])
            else:
                def f(x,valid,memory=memory,source_valid=source_valid,block=block):return self.block(block,x,valid,memory,source_valid)
                x=checkpoint(f,x,valid,use_reentrant=False) if self.training and self.checkpointing else f(x,valid)
        return self.backbone.final_norm(x),new
    def forward(self,batch):
        labels=batch['target'];bos=torch.full((len(labels),1),batch['bos'],device=labels.device,dtype=torch.long)
        target_in=torch.cat([bos,labels[:,:-1].clamp_min(0)],1);target_valid=torch.cat([torch.ones_like(bos,dtype=torch.bool),labels[:,:-1]!=-100],1)
        if self.native:h,_=self.hidden(target_in,target_valid,batch['source'],batch['source_valid'])
        else:
            ids=torch.cat([batch['source'],target_in],1);valid=torch.cat([batch['source_valid'],target_valid],1)
            h,_=self.hidden(ids,valid);h=h[:,-labels.shape[1]:]
        keep=labels!=-100;h=h[keep];y=labels[keep];loss=h.new_zeros((),dtype=torch.float32)
        # Checkpoint only projection/loss: avoid storing source×vocabulary logits or every target logit for backward.
        for start in range(0,len(y),self.cfg.get('projection_chunk',128)):
            hh=h[start:start+self.cfg.get('projection_chunk',128)];yy=y[start:start+self.cfg.get('projection_chunk',128)]
            def ce(hh,yy):
                logits=self.backbone.project(hh)
                with torch.autocast(device_type=hh.device.type,enabled=False):
                    return F.cross_entropy(logits.float(),yy,reduction='sum',label_smoothing=self.cfg.get('label_smoothing',.05))
            loss=loss+(checkpoint(ce,hh,yy,use_reentrant=False) if self.training else ce(hh,yy))
        return loss/max(1,len(y))
    def report(self):return {'total':sum(p.numel() for p in self.parameters()),'trainable':sum(p.numel() for p in self.parameters() if p.requires_grad),'architecture':self.backbone.config.architecture,'method':self.cfg['method']}

def collate(rows,cache,device='cpu',target_override=None):
    special=cache['special'];B=len(rows);S=cache['source_tokens'];T=max(len((target_override or {}).get(r['id'],r['target']) or []) for r in rows)
    source=torch.full((B,S),special['pad'],dtype=torch.long,device=device);valid=torch.zeros_like(source,dtype=torch.bool)
    target=torch.full((B,max(1,T)),-100,dtype=torch.long,device=device)
    for i,r in enumerate(rows):
        src=r['source'][:S];source[i,:len(src)]=torch.tensor(src,device=device);valid[i,:len(src)]=True
        yy=(target_override or {}).get(r['id'],r['target'])
        if yy:target[i,:len(yy)]=torch.tensor(yy,device=device)
    return {'source':source,'source_valid':valid,'target':target,'bos':special['bos']}

def reorder_state(state,index):
    out={}
    for k,v in state.items():
        if k in ('self','cross'):out[k]=[(a.index_select(0,index),b.index_select(0,index)) for a,b in v]
        elif k=='rnn':out[k]=tuple(t.index_select(1,index) for t in v)
        elif isinstance(v,torch.Tensor):out[k]=v.index_select(0,index)
        else:out[k]=v
    return out
def ban_ngrams(logits,sequences,n):
    if n<=0:return
    for i,seq in enumerate(sequences):
        if len(seq)<n-1:continue
        prefix=tuple(seq[-(n-1):]) if n>1 else ()
        banned={seq[j+n-1] for j in range(len(seq)-n+1) if tuple(seq[j:j+n-1])==prefix}
        if banned:logits[i,list(banned)]=float('-inf')
@torch.inference_mode()
def generate(model,rows,cache,device,decode):
    model.eval();special=cache['special'];batch=collate(rows,cache,device);beams=decode.get('beams',1);B=len(rows)
    src=batch['source'].repeat_interleave(beams,0);sv=batch['source_valid'].repeat_interleave(beams,0)
    bos=torch.full((B*beams,1),special['bos'],device=device,dtype=torch.long)
    ids=bos if model.native else torch.cat([src,bos],1)
    valid=torch.ones_like(bos,dtype=torch.bool) if model.native else torch.cat([sv,torch.ones_like(bos,dtype=torch.bool)],1)
    amp=device.type=='cuda';sequences=[[] for _ in range(B*beams)];scores=torch.full((B,beams),float('-inf'),device=device);scores[:,0]=0
    finished=torch.zeros(B*beams,dtype=torch.bool,device=device);lengths=torch.zeros(B*beams,device=device);state=None
    max_new=decode.get('max_new_tokens',cache['target_tokens']);min_new=decode.get('min_new_tokens',48)
    for step in range(max_new):
        with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=amp):
            h,state=model.hidden(ids,valid,src if model.native else None,sv if model.native else None,state,True)
            logits=model.backbone.project(h[:,-1]).float()
        logits[:,special['pad']]=float('-inf');logits[:,special['bos']]=float('-inf')
        if cache.get('generation_blocked_ids'):logits[:,cache['generation_blocked_ids']]=float('-inf')
        if step<min_new:logits[:,special['eos']]=float('-inf')
        ban_ngrams(logits,sequences,decode.get('no_repeat_ngram',4))
        logits[finished]=float('-inf');logits[finished,special['eos']]=0
        lp=logits.log_softmax(-1);V=lp.shape[-1]
        candidate=(scores[:,:,None]+lp.reshape(B,beams,V)).reshape(B,-1)
        next_scores,choice=candidate.topk(beams,dim=1);parent=(choice//V+torch.arange(B,device=device)[:,None]*beams).reshape(-1);token=(choice%V).reshape(-1)
        state=reorder_state(state,parent);oldfinished=finished.index_select(0,parent);lengths=lengths.index_select(0,parent)+(~oldfinished).float()
        sequences=[sequences[p]+([] if oldfinished[j].item() else [int(token[j])]) for j,p in enumerate(parent.tolist())]
        finished=oldfinished|(token==special['eos']);scores=next_scores
        if bool(finished.all()):break
        ids=token[:,None];valid=torch.ones_like(ids,dtype=torch.bool)
    normalized=scores/lengths.reshape(B,beams).clamp_min(1).pow(decode.get('length_penalty',.8))
    winners=normalized.argmax(-1).tolist();out=[]
    for i,k in enumerate(winners):
        seq=sequences[i*beams+k]
        if special['eos'] in seq:seq=seq[:seq.index(special['eos'])]
        out.append(seq)
    return out
'''
(CODE / 's6_model.py').write_text(module_source, encoding='utf-8')
print('Wrote s6_model.py')

In [ ]:
# s6_run.py
module_source = r'''"""torchrun worker: genuine two-GPU DDP, resume, BERTScore selection and generation."""
import argparse,contextlib,gc,json,math,os,random,time
from pathlib import Path
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader,DistributedSampler
from tokenizers import Tokenizer
from s6_data import atomic_json,read_json,file_hash,digest
from s6_model import load_parent,Summarizer,collate,generate,merged_adapters

def atomic_save(p,x):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);t=p.with_name(p.name+'.tmp');torch.save(x,t);t.replace(p)
def initialize():
    world=int(os.environ.get('WORLD_SIZE',1));rank=int(os.environ.get('RANK',0));local=int(os.environ.get('LOCAL_RANK',0))
    device=torch.device(f'cuda:{local}' if torch.cuda.is_available() else 'cpu')
    if device.type=='cuda':torch.cuda.set_device(device)
    if world>1:
        opts={'device_id':device} if device.type=='cuda' else {}
        dist.init_process_group('nccl' if device.type=='cuda' else 'gloo',**opts)
    return rank,world,device
def barrier(device):
    if dist.is_initialized():dist.barrier(device_ids=[device.index] if device.type=='cuda' else None)
def all_finite(value,device):
    flag=torch.isfinite(value.detach()).all().to(device=device,dtype=torch.int32)
    if dist.is_initialized():dist.all_reduce(flag,op=dist.ReduceOp.MIN)
    return bool(flag.item())
def finish_update(model,optimizer,scaler,params,device,stage):
    """Unscale once at the effective-batch boundary; all ranks step or skip together."""
    scaler.unscale_(optimizer)
    gradients=[p.grad for p in params if p.grad is not None]
    if not gradients:raise RuntimeError('No trainable gradients were produced.')
    norm=torch.nn.utils.get_total_norm(gradients,error_if_nonfinite=False)
    if not bool(torch.isfinite(norm)) and all(bool(torch.isfinite(g).all()) for g in gradients):
        # A float32 norm reduction itself can overflow even when every gradient element is finite.
        norm=torch.linalg.vector_norm(torch.stack([torch.linalg.vector_norm(g.detach().double()) for g in gradients]))
    if not all_finite(norm,device):
        bad=[n for n,p in model.named_parameters() if p.grad is not None and not bool(torch.isfinite(p.grad).all())][:8]
        previous=float(scaler.get_scale());new_scale=previous*.5
        print(json.dumps({'event':'amp_gradient_overflow','stage':stage,'rank':dist.get_rank() if dist.is_initialized() else 0,
                          'scale_before':previous,'scale_after':new_scale,'nonfinite_gradient_parameters':bad,
                          'optimizer_step':'skipped on all ranks; parameters unchanged'}),flush=True)
        if not scaler.is_enabled():raise FloatingPointError(f'{stage}: nonfinite unscaled gradients with AMP disabled: {bad}')
        # Reset per-optimizer scaler state without updating weights, even if only a peer rank overflowed.
        scaler.update(new_scale=new_scale);optimizer.zero_grad(set_to_none=True)
        return float('inf'),False
    torch.nn.utils.clip_grads_with_norm_(params,1.,norm)
    scaler.step(optimizer);scaler.update()
    return float(norm),True
def preflight_update(wrapped,model,optimizer,scaler,params,batch,device,amp,stage,max_attempts=12):
    for attempt in range(max_attempts):
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=amp):loss=wrapped(batch)
        if not all_finite(loss,device):
            print(json.dumps({'event':'nonfinite_forward_loss','stage':stage,'rank':dist.get_rank() if dist.is_initialized() else 0,
                              'local_loss':str(float(loss.detach())),'amp_scale':float(scaler.get_scale())}),flush=True)
            raise FloatingPointError(f'{stage}: forward loss is nonfinite; this is not a loss-scaler-only overflow.')
        scaler.scale(loss).backward()
        # Watch a matrix with a nonzero gradient, not an arbitrary softmax bias whose gradient can be exactly zero.
        watch=next((p for p in params if p.grad is not None and p.ndim>1 and bool(torch.isfinite(p.grad).all()) and bool(p.grad.ne(0).any())),None)
        before=watch.detach().clone() if watch is not None else None
        norm,stepped=finish_update(model,optimizer,scaler,params,device,stage)
        if stepped:
            if watch is None or torch.equal(before,watch.detach()):raise ValueError('Preflight optimizer produced no detectable trainable matrix update.')
            return {'loss':float(loss.detach()),'gradient_norm':norm,'overflow_retries':attempt,'amp_scale':float(scaler.get_scale())}
    raise FloatingPointError(f'{stage}: gradients stayed nonfinite after {max_attempts} attempts. Last AMP scale={scaler.get_scale()}; inspect the printed parameter diagnostics.')
def build(cfg,cache,device):
    parent=load_parent(cfg['parent'],cache['tokenizer_sha256'],cfg.get('allow_tiny',False))
    if parent.config.architecture!=cfg['architecture']:raise ValueError('Wrong original architecture family.')
    total=cache['source_tokens']+cache['target_tokens']+cfg.get('prompt_tokens',0)+1
    if not cfg['architecture']=='encoder_decoder' and total>parent.config.max_positions:raise ValueError(f'Source+target+prompt ({total}) exceeds original RoPE limit {parent.config.max_positions}.')
    if cfg['architecture']=='encoder_decoder' and max(cache['source_tokens'],cache['target_tokens'])>parent.config.max_positions:raise ValueError('Native encoder/decoder context exceeds Phase-1 position limit.')
    return Summarizer(parent,cfg).to(device)
def predict(model,rows,cache,device,decode,batch_size):
    with merged_adapters(model):return _predict(model,rows,cache,device,decode,batch_size)
def _predict(model,rows,cache,device,decode,batch_size):
    tok=Tokenizer.from_file(cache['tokenizer_path']);out=[];start=time.perf_counter()
    for a in range(0,len(rows),batch_size):
        local=rows[a:a+batch_size];tokens=generate(model,local,cache,device,decode)
        for r,t in zip(local,tokens):
            text=tok.decode(t,skip_special_tokens=True).strip()
            if not text and r['split']!='dev':raise ValueError(f'{r["id"]}: empty neural summary; inspect training and special tokens before submission.')
            out.append({'id':r['id'],'summary':text})
        if (a//batch_size+1)%25==0:print(json.dumps({'generated':min(len(rows),a+batch_size),'total':len(rows),'seconds':round(time.perf_counter()-start,1)}),flush=True)
    return out
def main():
    ap=argparse.ArgumentParser();ap.add_argument('--config',required=True);ap.add_argument('--action',choices=['train','preflight','generate'],default='train');ap.add_argument('--split',default='test');ap.add_argument('--output');ap.add_argument('--limit',type=int);ap.add_argument('--checkpoint',default='best.pt');ap.add_argument('--decode-file');args=ap.parse_args()
    cfg=read_json(args.config);cache=torch.load(cfg['cache'],map_location='cpu',weights_only=False);rank,world,device=initialize()
    random.seed(cfg['seed']+rank);np.random.seed(cfg['seed']+rank);torch.manual_seed(cfg['seed'])
    if device.type=='cuda':torch.backends.cudnn.benchmark=True
    run=Path(cfg['run']);run.mkdir(parents=True,exist_ok=True)
    lineage={'parent_sha256':file_hash(cfg['parent']),'tokenizer_sha256':cache['tokenizer_sha256'],'data_fingerprint':cache['fingerprint'],'config':cfg,'world_size':world,
             'code_sha256':{p.name:file_hash(p) for p in Path(__file__).parent.glob('*.py')}}
    if cfg['method']=='kd':lineage['kd_cache_sha256']=file_hash(cfg['kd_cache']) if Path(cfg['kd_cache']).is_file() else None
    signature=digest(lineage);model=build(cfg,cache,device);report=model.report()
    if args.action=='generate':
        saved=torch.load(run/args.checkpoint,map_location='cpu',weights_only=False)
        if saved['signature']!=signature:raise ValueError('Checkpoint provenance/config mismatch.')
        model.load_state_dict(saved['model'],strict=True)
        rows=sorted([r for r in cache['rows'] if r['split']==args.split],key=lambda r:digest([cfg['seed'],r['id']]) if args.split=='train' else r['id'])
        if args.limit:rows=rows[:args.limit]
        local=rows[rank::world];decode=read_json(args.decode_file) if args.decode_file else cfg['decode'];pred=predict(model,local,cache,device,decode,cfg['generation_batch'])
        output=Path(args.output);atomic_json(str(output)+f'.rank{rank}.json',pred);barrier(device)
        if rank==0:
            pred=sum([read_json(str(output)+f'.rank{i}.json') for i in range(world)],[]);byid={r['id']:r['summary'] for r in pred}
            if len(byid)!=len(rows) or set(byid)!={r['id'] for r in rows}:raise ValueError('Distributed generation IDs missing or duplicated.')
            atomic_json(output,[{'id':r['id'],'summary':byid[r['id']]} for r in rows])
        barrier(device)
        if dist.is_initialized():dist.destroy_process_group()
        return
    train=[r for r in cache['rows'] if r['split']=='train'];dev=sorted([r for r in cache['rows'] if r['split']=='dev'],key=lambda r:digest([cfg['seed'],r['id']]))[:cfg['dev_probe']]
    teacher={}
    if cfg['method']=='kd' and args.action=='train':
        kd=read_json(cfg['kd_cache']);allowed={r['id'] for r in train}
        if kd['data_fingerprint']!=cache['fingerprint'] or not set(kd['targets'])<=allowed:raise ValueError('KD cache must contain training IDs from this exact split.')
        if kd['teacher_checkpoint_sha256']!=file_hash(kd['teacher_checkpoint']):raise ValueError('KD teacher changed; rebuild KD cache.')
        teacher=kd['targets']
    wrapped=DDP(model,device_ids=[device.index] if device.type=='cuda' else None,broadcast_buffers=False) if world>1 else model
    params=[p for p in model.parameters() if p.requires_grad];optimizer=torch.optim.AdamW(params,lr=cfg['lr'],weight_decay=cfg['weight_decay'],**({'fused':True} if device.type=='cuda' else {}))
    amp=device.type=='cuda';initial_scale=1024.
    if args.action=='train' and (run/'preflight.json').is_file():
        initial_scale=min(initial_scale,float(read_json(run/'preflight.json').get('amp_scale',initial_scale)))
    scaler=torch.amp.GradScaler(device.type,init_scale=initial_scale,enabled=amp)
    last=run/'last.pt';best=-float('inf');best_epoch=0;start_epoch=0;updates=0;stale=0;elapsed_base=0.;start=time.perf_counter()
    if args.action=='train' and last.exists():
        saved=torch.load(last,map_location='cpu',weights_only=False)
        if saved['signature']!=signature:raise ValueError('Resume config/input differs; use a new PROJECT or experiment name.')
        model.load_state_dict(saved['model'],strict=True);optimizer.load_state_dict(saved['optimizer']);scaler.load_state_dict(saved['scaler'])
        start_epoch=saved['epoch'];updates=saved['updates'];best=saved['best'];best_epoch=saved['best_epoch'];stale=saved['stale'];elapsed_base=saved['elapsed_hours']
        state=saved['rng'][rank];random.setstate(state['python']);np.random.set_state(state['numpy']);torch.set_rng_state(state['torch'])
        if device.type=='cuda':torch.cuda.set_rng_state(state['cuda'],device)
    if args.action=='preflight':
        model.train();batch=collate([max(train,key=lambda r:len(r['target']))],cache,device)
        # Measure the full configured context and a real long target, not a reduced smoke-test sequence.
        timings=[];retry_count=0
        for i in range(3):
            if amp:torch.cuda.synchronize()
            t=time.perf_counter()
            update=preflight_update(wrapped,model,optimizer,scaler,params,batch,device,amp,f'full_context_{i+1}')
            retry_count+=update['overflow_retries']
            if amp:torch.cuda.synchronize()
            timings.append(time.perf_counter()-t)
        checksum=torch.stack([p.detach().float().sum() for p in params]).sum()
        if world>1:
            lo=checksum.clone();hi=checksum.clone();dist.all_reduce(lo,op=dist.ReduceOp.MIN);dist.all_reduce(hi,op=dist.ReduceOp.MAX)
            if not torch.allclose(lo,hi,rtol=1e-6,atol=1e-5):raise ValueError('DDP ranks diverged.')
        result={'world_size':world,'finite_update':True,'full_source_tokens':cache['source_tokens'],'target_tokens':int((batch['target']!=-100).sum()),'seconds_per_microbatch_per_rank':sum(timings[1:])/2,'peak_reserved_gib_per_rank':torch.cuda.max_memory_reserved()/1024**3 if amp else None,'parameters':report,'amp_scale':float(scaler.get_scale()),'overflow_retries':retry_count}
        if rank==0:atomic_json(run/'preflight.json',result);print(json.dumps(result),flush=True)
        barrier(device)
        if dist.is_initialized():dist.destroy_process_group()
        return
    sampler=DistributedSampler(train,num_replicas=world,rank=rank,shuffle=True,seed=cfg['seed'])
    loader=DataLoader(train,batch_size=cfg['micro_batch'],sampler=sampler,num_workers=0,collate_fn=lambda rows:rows)
    total_updates=max(1,math.ceil(len(loader)/cfg['accumulation'])*cfg['epochs']);skips=0;consecutive_skips=0
    if rank==0:print(json.dumps({'experiment':cfg['name'],'documents':len(train),'effective_batch':world*cfg['micro_batch']*cfg['accumulation'],'parameters':report}),flush=True)
    if stale>=cfg['patience']:start_epoch=cfg['epochs']
    for epoch in range(start_epoch,cfg['epochs']):
        sampler.set_epoch(epoch);model.train();optimizer.zero_grad(set_to_none=True);running=0
        for i,rows in enumerate(loader):
            # A deterministic Bernoulli choice gives E[L] = .5 gold NLL + .5 sequence-KD NLL on cached training cases.
            override={r['id']:teacher[r['id']] for r in rows if r['id'] in teacher and int(digest([cfg['seed'],epoch,r['id']])[:8],16)%2==0}
            batch=collate(rows,cache,device,override);group_start=(i//cfg['accumulation'])*cfg['accumulation'];divisor=min(cfg['accumulation'],len(loader)-group_start)
            sync=(i+1)%cfg['accumulation']==0 or i+1==len(loader);ctx=wrapped.no_sync() if world>1 and not sync else contextlib.nullcontext()
            with ctx:
                with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=amp):loss=wrapped(batch)
                if not all_finite(loss,device):raise FloatingPointError(f'Nonfinite forward loss at epoch={epoch+1} batch={i+1}.')
                scaler.scale(loss/divisor).backward()
            running+=float(loss.detach())
            if sync:
                progress=updates/total_updates;warm=cfg['warmup_fraction'];factor=min(1.,(updates+1)/max(1,total_updates*warm)) if progress<warm else .1+.9*.5*(1+math.cos(math.pi*(progress-warm)/max(1e-8,1-warm)))
                for group in optimizer.param_groups:group['lr']=cfg['lr']*factor
                norm,stepped=finish_update(model,optimizer,scaler,params,device,f'epoch_{epoch+1}_batch_{i+1}')
                skips+=not stepped;consecutive_skips=0 if stepped else consecutive_skips+1
                optimizer.zero_grad(set_to_none=True);updates+=int(stepped)
                if consecutive_skips>=12:raise FloatingPointError('Persistent gradient overflow; inspect the logged parameters.')
                if rank==0 and stepped and updates%cfg['log_every']==0:print(json.dumps({'epoch':epoch+1,'batch':i+1,'updates':updates,'nll_smoothed':running/(i+1),'gradient_norm':float(norm),'lr':optimizer.param_groups[0]['lr'],'amp_skips':skips,'elapsed_hours':elapsed_base+(time.perf_counter()-start)/3600}),flush=True)
        # No references on test ever enter this branch. All ranks generate disjoint validation IDs.
        model.eval();local=predict(model,dev[rank::world],cache,device,cfg['decode'],cfg['generation_batch'])
        atomic_json(run/f'dev_epoch_{epoch+1}.rank{rank}.json',local);barrier(device)
        decision=[None]
        if rank==0:
            pred=sum([read_json(run/f'dev_epoch_{epoch+1}.rank{r}.json') for r in range(world)],[]);byid={r['id']:r['summary'] for r in pred}
            if len(byid)!=len(dev):raise ValueError('Incomplete validation predictions.')
            from s6_metrics import score,release
            metric=score([byid[r['id']] for r in dev],[r['summary'] for r in dev],device)
            release()
            value=metric['bertscore_f1'];improved=value>best+cfg['min_delta']
            if improved:best=value;best_epoch=epoch+1;stale=0
            else:stale+=1
            decision[0]={'best':best,'best_epoch':best_epoch,'stale':stale,'improved':improved,'metrics':metric}
            atomic_json(run/f'metrics_epoch_{epoch+1}.json',decision[0]);print(json.dumps({'epoch':epoch+1,**decision[0],'elapsed_hours':elapsed_base+(time.perf_counter()-start)/3600}),flush=True)
        if world>1:dist.broadcast_object_list(decision,src=0,device=device)
        d=decision[0];best=d['best'];best_epoch=d['best_epoch'];stale=d['stale']
        rng={'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),'cuda':torch.cuda.get_rng_state(device) if amp else None};rngs=[None]*world
        if world>1:dist.all_gather_object(rngs,rng)
        else:rngs=[rng]
        if rank==0:
            elapsed=elapsed_base+(time.perf_counter()-start)/3600
            saved={'model':model.state_dict(),'signature':signature,'lineage':lineage,'epoch':epoch+1,'updates':updates,'best':best,'best_epoch':best_epoch,'stale':stale,'elapsed_hours':elapsed,'parameters':report,'rng':rngs}
            if d['improved']:
                atomic_save(run/'best.pt',saved);atomic_json(run/'best_metrics.json',d)
            atomic_save(last,{**saved,'optimizer':optimizer.state_dict(),'scaler':scaler.state_dict()})
        barrier(device)
        if stale>=cfg['patience']:break
    if dist.is_initialized():dist.destroy_process_group()
if __name__=='__main__':main()
'''
(CODE/'s6_run.py').write_text(module_source, encoding='utf-8')
print('Wrote s6_run.py')

In [ ]:
# s6_workflow.py
module_source = r'''"""Notebook orchestration; saves every completed submission and ranks on held-out data only."""
import gc,json,math,os,shutil,subprocess,sys,time,zipfile
from pathlib import Path
import torch
from tokenizers import Tokenizer
from s6_data import FILES,atomic_json,read_json,file_hash,digest

def find_phase1(explicit=None):
    if explicit:
        p=Path(explicit)
        if not (p/'tokenizer.json').is_file():raise FileNotFoundError(f'No tokenizer.json under {p}')
        return p
    preferred=Path('/kaggle/input/datasets/hardik4tiwari/phase-1-outputs/capstone_phase1')
    candidates=[preferred] if preferred.is_dir() else sorted({p.parent for p in Path('/kaggle/input').rglob('tokenizer.json') if p.parent.name=='capstone_phase1'})
    if len(candidates)!=1:raise FileNotFoundError('Set PHASE1_ROOT to the unique original capstone_phase1 directory. Attach the Phase-1 outputs dataset.')
    return candidates[0]
def obtain_data(project,local_root=None,revision=None):
    if not local_root:
        candidates=sorted({p for p in Path('/kaggle/input').rglob('Phase_6_Summ') if (p/FILES[0]).is_file() and (p/FILES[1]).is_file()})
        if len(candidates)==1:local_root=candidates[0]
        elif len(candidates)>1:raise ValueError('Multiple Phase-6 datasets: set DATA_ROOT explicitly.')
    if local_root:
        source=Path(local_root)
        if (source/'Phase_6_Summ').is_dir():source/='Phase_6_Summ'
        if any(not (source/f).is_file() for f in FILES):raise FileNotFoundError('Need Phase_6_Summ/Train/train.json and Test/test.json.')
        atomic_json(Path(project)/'data_source.json',{'kind':'attached_official_phase6','root':str(source),'sha256':{n:file_hash(source/n) for n in FILES}})
        return str(source)
    from huggingface_hub import HfApi,hf_hub_download
    token=os.environ.get('HF_TOKEN')
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token=UserSecretsClient().get_secret('HF_TOKEN')
        except Exception:token=None
    root=Path(project)/'data';manifest=root/'source.json';old=read_json(manifest) if manifest.exists() else None
    repo='Exploration-Lab/CS-781-Capstone'
    try:
        commit=old['revision'] if old else HfApi(token=token).dataset_info(repo,revision=revision).sha
        if old and revision and revision!=commit:raise ValueError('Pinned revision differs. Choose a new PROJECT.')
        for name in FILES:
            target=root/name
            if target.exists() and old and old.get('sha256',{}).get(name)==file_hash(target):continue
            source=hf_hub_download(repo,'Phase_6_Summ/'+name,repo_type='dataset',revision=commit,token=token)
            target.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(source,target)
        atomic_json(manifest,{'repo':repo,'revision':commit,'sha256':{n:file_hash(root/n) for n in FILES}})
    except Exception as e:raise RuntimeError('Attach official Phase_6_Summ data, or accept its Hugging Face terms, enable Kaggle Internet and add an HF_TOKEN secret.') from e
    return str(root)
def configurations(project,phase1,cache_path,settings):
    project=Path(project);phase1=Path(phase1);cache=torch.load(cache_path,map_location='cpu',weights_only=False)
    specs=[('01_encoder_decoder_full','encoder_decoder','full',settings['full_epochs'],8e-5),
           ('02_decoder_lorma','decoder','lorma',settings['peft_epochs'],2e-4),
           ('03_local_gqa_adapter','local_gqa','adapter',settings['peft_epochs'],3e-4),
           ('04_hybrid_lora','hybrid','lora',settings['peft_epochs'],2e-4),
           ('05_decoder_prompt','decoder','prompt',settings['peft_epochs'],3e-3),
           ('06_decoder_student_kd','decoder','kd',settings['student_epochs'],8e-5)]
    result=[]
    for name,arch,method,epochs,lr in specs:
        parent=phase1/'runs'/('main_'+arch)/'best_model.pt'
        if not parent.is_file():raise FileNotFoundError(f'Missing original Phase-1 parent: {parent}')
        cfg={'name':name,'architecture':arch,'method':method,'parent':str(parent),'cache':str(cache_path),
             'run':str(project/'runs'/name),'seed':settings['seed'],'epochs':epochs,'lr':lr,
             'source_tokens':cache['source_tokens'],
             'weight_decay':.01,'warmup_fraction':.06,'patience':settings['patience'],'min_delta':.0002,
             'micro_batch':settings['micro_batch'],'accumulation':settings['accumulation'],
             'generation_batch':settings['generation_batch'],'dev_probe':settings['dev_probe'],
             'rank':16,'alpha':32,'adapter_dim':64,'prompt_tokens':32 if method=='prompt' else 0,
             'globalize_local':arch=='local_gqa','checkpointing':True,'dropout':.05,
             'projection_chunk':128,'label_smoothing':.05,'log_every':25,
             'kd_cache':str(project/'kd_cache.json'),
             'decode':{'beams':1,'min_new_tokens':min(64,cache['target_tokens']//4),
                       'max_new_tokens':cache['target_tokens'],'length_penalty':.8,'no_repeat_ngram':4}}
        path=project/'configs'/f'{name}.json'
        if path.exists() and read_json(path)!=cfg:raise ValueError(f'Config changed for {name}. Use a new PROJECT to preserve resume integrity.')
        atomic_json(path,cfg);result.append(str(path))
    return result
def launch(config,action='train',world=2,**kwargs):
    module=Path(__file__).parent/'s6_run.py'
    command=[sys.executable,'-m','torch.distributed.run','--standalone',f'--nproc_per_node={world}',str(module),'--config',str(config),'--action',action]
    for key,value in kwargs.items():command+=['--'+key.replace('_','-'),str(value)]
    env=os.environ.copy();env['TOKENIZERS_PARALLELISM']='false';env['OMP_NUM_THREADS']='2';env['PYTHONUNBUFFERED']='1'
    env['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True';env['PYTHONPATH']=str(module.parent)+os.pathsep+env.get('PYTHONPATH','')
    cfg=read_json(config);log=Path(cfg['run'])/(action+'.log');log.parent.mkdir(parents=True,exist_ok=True)
    with log.open('a',encoding='utf-8') as handle:
        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
        for line in process.stdout:print(line,end='');handle.write(line);handle.flush()
        status=process.wait()
    if status:raise RuntimeError(f'{action} failed for {cfg["name"]}; inspect {log}. Completed runs remain saved.')
def forecast(configs,world=2):
    rows=[];total=0
    for path in configs:
        cfg=read_json(path);p=Path(cfg['run'])/'preflight.json';pre=read_json(p);cache=torch.load(cfg['cache'],map_location='cpu',weights_only=False)
        n=sum(r['split']=='train' for r in cache['rows']);batches=math.ceil(math.ceil(n/world)/cfg['micro_batch'])
        # Preflight uses batch=1 with longest target, and synchronizes every step (conservative for no_sync accumulation).
        hours=pre['seconds_per_microbatch_per_rank']*cfg['micro_batch']*batches*cfg['epochs']/3600
        total+=hours;rows.append({'name':cfg['name'],'full_epoch_training_hours_conservative':hours/cfg['epochs'],'max_epochs':cfg['epochs'],'training_hours_conservative':hours,'peak_preflight_gib':pre['peak_reserved_gib_per_rank']})
    print(json.dumps(rows,indent=2));print(f'Training-only conservative forecast: {total:.2f} h. Excludes preprocessing, BERTScore, generation, teacher cache and disk I/O; early stopping can reduce it.')
    return rows
def kd_cache(project,teacher_config,world=2,limit=2048):
    project=Path(project);cfg=read_json(teacher_config);cache=torch.load(cfg['cache'],map_location='cpu',weights_only=False)
    checkpoint=Path(cfg['run'])/'best.pt';dest=project/'kd_cache.json';identity={'data_fingerprint':cache['fingerprint'],'teacher_checkpoint':str(checkpoint),'teacher_checkpoint_sha256':file_hash(checkpoint),'decode':cfg['decode'],'limit':limit}
    if dest.exists():
        old=read_json(dest)
        if any(old.get(k)!=v for k,v in identity.items()):raise ValueError('KD teacher/cache changed; use a new PROJECT.')
        return dest
    output=project/'teacher_train_predictions.json'
    # Only training source documents are decoded; no dev/test IDs are permitted in the KD cache.
    launch(teacher_config,'generate',world=world,split='train',output=output,limit=limit)
    tok=Tokenizer.from_file(cache['tokenizer_path']);allowed={r['id'] for r in cache['rows'] if r['split']=='train'};targets={}
    for r in read_json(output):
        if r['id'] not in allowed:raise ValueError('Held-out ID entered KD cache.')
        targets[r['id']]=tok.encode(r['summary'],add_special_tokens=False).ids[:cache['target_tokens']-1]+[cache['special']['eos']]
    atomic_json(dest,{**identity,'targets':targets,'objective':'Stochastic 50:50 gold vs greedy teacher sequence targets on cached training IDs; remaining train IDs use gold.'});return dest
def pack_prediction(project,name,pred_path,cache_path):
    cache=torch.load(cache_path,map_location='cpu',weights_only=False);expected={r['id'] for r in cache['rows'] if r['split']=='test'}
    pred=read_json(pred_path)
    if not isinstance(pred,list) or len(pred)!=len(expected):raise ValueError('Wrong number of test predictions.')
    if len({r['id'] for r in pred})!=len(pred) or {r['id'] for r in pred}!=expected:raise ValueError('Submission IDs mismatch.')
    if any(set(r)!={'id','summary'} or not isinstance(r['summary'],str) or not r['summary'].strip() for r in pred):raise ValueError('Invalid summary schema.')
    folder=Path(project)/'submissions'/name;atomic_json(folder/'prediction.json',pred);archive=folder.with_suffix('.zip')
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:z.write(folder/'prediction.json','prediction.json')
    with zipfile.ZipFile(archive) as z:
        if z.namelist()!=['prediction.json']:raise ValueError('Incorrect ZIP root.')
    return {'name':name,'submission_zip':str(archive),'prediction_sha256':digest(sorted(pred,key=lambda r:r['id'])),'test_documents':len(pred)}
def run_all(project,configs,settings,world=2):
    project=Path(project);results=[];begin=time.perf_counter()
    for path in configs:
        cfg=read_json(path);run=Path(cfg['run'])
        completed=run/'completed.json'
        identity={'config_sha256':digest(cfg),'cache_sha256':file_hash(cfg['cache']),'parent_sha256':file_hash(cfg['parent']),
                  'code_sha256':{p.name:file_hash(p) for p in Path(__file__).parent.glob('*.py')},'world_size':world}
        if completed.exists():
            old=read_json(completed);item=old['result']
            if old['identity']!=identity or old['checkpoint_sha256']!=file_hash(run/'best.pt'):raise ValueError('Completed run inputs changed. Use a new PROJECT.')
            if not Path(item['submission_zip']).is_file():raise FileNotFoundError('Completed submission archive was removed.')
            results.append(item);atomic_json(project/'results.json',results);print('Reusing completed submission:',cfg['name']);continue
        if cfg['method']=='kd':
            teachers=[p for p in configs if read_json(p)['method']!='kd' and (Path(read_json(p)['run'])/'best_metrics.json').is_file()]
            teacher=max(teachers,key=lambda p:read_json(Path(read_json(p)['run'])/'best_metrics.json')['best'])
            kd_cache(project,teacher,world,settings['kd_cases'])
        launch(path,'train',world)
        # Choose greedy vs beam-2 using the same held-out probe; never inspect test references or leaderboard scores.
        cache=torch.load(cfg['cache'],map_location='cpu',weights_only=False)
        probe=sorted([r for r in cache['rows'] if r['split']=='dev'],key=lambda r:r['id'])[:settings['decode_tune_cases']]
        # A generation config changes decode only. Checkpoint signature must stay tied to original training config:
        # candidate selection is implemented outside worker using a local temporary model, then final inference uses chosen decode override.
        tuning=tune_decode(path,probe,cache,settings.get('tune_beams',True))
        atomic_json(run/'decode_selection.json',tuning)
        # Generate with an explicit decoding override file; it does not mutate the immutable training signature.
        decode_path=run/'selected_decode.json';atomic_json(decode_path,tuning['decode'])
        pred_path=run/'test_predictions.json';launch(path,'generate',world,split='test',output=pred_path,decode_file=decode_path)
        dev_path=run/'full_dev_predictions.json';launch(path,'generate',world,split='dev',output=dev_path,decode_file=decode_path)
        dev={r['id']:r['summary'] for r in read_json(dev_path)};held=[r for r in cache['rows'] if r['split']=='dev']
        from s6_metrics import score,release
        metrics=score([dev[r['id']] for r in held],[r['summary'] for r in held],torch.device('cuda:0' if torch.cuda.is_available() else 'cpu'));release()
        atomic_json(run/'full_dev_metrics.json',metrics)
        item={**pack_prediction(project,cfg['name'],pred_path,cfg['cache']),'architecture':cfg['architecture'],'method':cfg['method'],'held_out':metrics,'best_epoch':read_json(run/'best_metrics.json')['best_epoch']}
        atomic_json(completed,{'identity':identity,'checkpoint_sha256':file_hash(run/'best.pt'),'result':item})
        results.append(item);atomic_json(project/'results.json',results)
        print(json.dumps({'completed':cfg['name'],'held_out':metrics,'elapsed_all_hours':(time.perf_counter()-begin)/3600}),flush=True)
    ranked=sorted(results,key=lambda x:x['held_out']['bertscore_f1'],reverse=True);hashes={r['prediction_sha256'] for r in results}
    atomic_json(project/'leaderboard.json',ranked)
    if ranked:shutil.copy2(ranked[0]['submission_zip'],project/'best_submission.zip')
    if len(hashes)<len(results):print('WARNING: Some models produced identical predictions; those do not count as distinct submissions. Inspect results before submission.')
    atomic_json(project/'compliance.json',{'completed':len(results),'distinct_prediction_files':len(hashes),'families':sorted({r['architecture'] for r in results}),'methods':sorted({r['method'] for r in results}),'satisfies_six_distinct_and_four_families':len(results)>=6 and len(hashes)>=6 and len({r['architecture'] for r in results})==4})
    return ranked
def tune_decode(path,rows,cache,enabled=True):
    # This parent process is otherwise CPU-only. All torchrun workers have exited before loading model + evaluation metric here.
    from s6_run import build,predict
    from s6_metrics import score,release
    cfg=read_json(path);device=torch.device('cuda:0' if torch.cuda.is_available() else 'cpu');model=build(cfg,cache,device)
    saved=torch.load(Path(cfg['run'])/'best.pt',map_location='cpu',weights_only=False);model.load_state_dict(saved['model'],strict=True);del saved
    candidates=[]
    for beam in ([1,2] if enabled else [1]):
        decode={**cfg['decode'],'beams':beam};pred=predict(model,rows,cache,device,decode,cfg['generation_batch'])
        metric=score([p['summary'] for p in pred],[r['summary'] for r in rows],device);release()
        candidates.append({'decode':decode,'metrics':metric})
    winner=max(candidates,key=lambda x:x['metrics']['bertscore_f1']);del model;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return {**winner,'candidates':candidates,'selection':'held-out decode-tuning subset; full dev is not an independent unbiased estimate'}
def archive_all(project):
    project=Path(project);output=project.parent/(project.name+'_all_outputs.zip')
    size=sum(p.stat().st_size for p in project.rglob('*') if p.is_file())
    if shutil.disk_usage(project.parent).free<size+512*1024**2:raise OSError('Not enough free disk to duplicate all outputs into ZIP. Individual submission ZIPs and checkpoints are already saved; download them separately or free unrelated disk space.')
    with zipfile.ZipFile(output,'w',zipfile.ZIP_DEFLATED,compresslevel=1) as z:
        for p in sorted(project.rglob('*')):
            if p.is_file() and '__pycache__' not in p.parts and not p.name.endswith('.tmp'):z.write(p,str(p.relative_to(project.parent)))
    return output
'''
(CODE / 's6_workflow.py').write_text(module_source, encoding='utf-8')
print('Wrote s6_workflow.py')

In [ ]:
import importlib, platform
sys.path.insert(0, str(CODE)); importlib.invalidate_caches()
from s6_data import prepare_data, read_json, atomic_json, file_hash
from s6_workflow import find_phase1, obtain_data, configurations, launch, forecast, run_all, archive_all
PHASE1_ROOT = find_phase1(PHASE1_ROOT)
TOKENIZER_PATH = PHASE1_ROOT / 'tokenizer.json'
data_root = obtain_data(PROJECT, DATA_ROOT, DATA_REVISION)
CACHE_PATH = prepare_data(data_root, TOKENIZER_PATH, PROJECT / 'token_cache', SETTINGS)
cache = torch.load(CACHE_PATH, map_location='cpu', weights_only=False)
audit_view = dict(cache['audit'])
train_audit = dict(audit_view['train_records'])
skipped = train_audit['skipped_rows']
train_audit.update(skipped_count=len(skipped), skipped_rows=skipped[:10])
audit_view['train_records'] = train_audit
print(json.dumps(audit_view, indent=2))
print('Complete row audit:', PROJECT/'token_cache'/'audit.json')
CONFIGS = configurations(PROJECT, PHASE1_ROOT, CACHE_PATH, SETTINGS)
for path in CONFIGS:
    cfg = read_json(path); print(cfg['name'], cfg['architecture'], cfg['method'])
atomic_json(PROJECT / 'environment.json', {'python': platform.python_version(), 'torch': torch.__version__,
    'gpu': [torch.cuda.get_device_name(i) for i in range(WORLD_SIZE)], 'settings': SETTINGS,
    'tokenizer_sha256': file_hash(TOKENIZER_PATH), 'cache_fingerprint': cache['fingerprint']})
subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=(PROJECT/'requirements_executed.txt').open('w'), check=True)


## Preflight before the long run

First confirm the official metric can load. This network is separate from every generator and optimizer. Internet is required only if its evaluation weights are not cached.

Each subsequent preflight launches **two actual GPU ranks**, runs three optimizer steps on a real long training target with the full configured source width, checks finite gradients/weight changes and matching rank checksums, and discards the modified model. It does not alter your Phase-1 weights or train checkpoints. The forecast covers training only; generation, BERTScore, KD target caching and I/O are additional. Memory checks here do not cover beam decoding or the BERTScore model, which is released after each evaluation.

In [ ]:
from s6_metrics import score, release
metric_smoke = score(['The appeal was dismissed.'], ['The appeal was dismissed.'], torch.device('cuda:0'))
print('Metric ready:', metric_smoke)
release()
for config_path in CONFIGS:
    launch(config_path, action='preflight', world=WORLD_SIZE)
    print('PASS actual full-context two-GPU preflight:', read_json(config_path)['name'])
training_forecast = forecast(CONFIGS, WORLD_SIZE)
atomic_json(PROJECT / 'training_forecast.json', training_forecast)


## Train, validate and save each submission immediately

Checkpoint selection uses a fixed held-out probe (up to 128 documents). Greedy versus beam-2 decoding is compared on up to 64 held-out documents, then all held-out documents are scored for reporting. These are **selection-set results, not an unbiased independent test estimate**. Nothing reads official test references.

The first five runs create candidates from all four original families. The sixth creates a smaller KD student using the strongest teacher. Maximum epochs are ceilings; patience=2 can stop earlier. No automatic training on the validation split or unvalidated train+dev refit is performed. `last.pt` resumes at the last completed epoch, including optimizer/scaler/RNG state. Completed submission files are reused on rerun.

Start with the defaults. If the measured forecast exceeds your available session, save outputs and continue another session with the same code/configuration and project state. Attach previous outputs and copy the **project directory** back to `/kaggle/working` before rerunning; do not redirect writable output to `/kaggle/input`. Shortening context/targets or training only a subset trades off coverage/completeness and is not presented as score-preserving.

In [ ]:
ranked = run_all(PROJECT, CONFIGS, SETTINGS, world=WORLD_SIZE)
print(json.dumps(ranked, indent=2))
print('Compliance:', json.dumps(read_json(PROJECT / 'compliance.json'), indent=2))


## Official evaluator cross-check for the selected model

The official evaluator is embedded unchanged. This checks your best model on held-out training documents only. It computes BERTScore with `lang='en'`, default model/layers, batch size 8, no baseline rescaling and no IDF, plus stemmed ROUGE-1 and the provided bitset-LCS ROUGE-L implementation. Its score can differ from ROUGE libraries using different tokenization or summary-level LCS.

In [ ]:
best = read_json(PROJECT / 'leaderboard.json')[0]
best_run = PROJECT / 'runs' / best['name']
truth = {r['id']: r['summary'] for r in cache['rows'] if r['split']=='dev'}
truth_path = PROJECT / 'held_out_truth.json'; atomic_json(truth_path, truth)
subprocess.check_call([sys.executable, str(CODE/'evaluate_phase6_official.py'),
                       '--truth', str(truth_path), '--prediction', str(best_run/'full_dev_predictions.json')])
print('Best held-out candidate:', best['name'])
print('Selected submission:', PROJECT / 'best_submission.zip')


## Download

Each individual submission ZIP has **only `prediction.json` at its root**, with exactly one nonempty generated summary per original test ID. `best_submission.zip` is a copy of the best held-out candidate’s ZIP. Use the individual ZIPs for the competition.

The separate all-outputs ZIP includes code, audits, selectors, caches, logs, checkpoints, validation predictions and every submission. It is for backup/download, not competition upload. Metric model weights outside the project are excluded. A free-space check prevents creating a large backup when disk is insufficient; individual submissions remain available.

In [ ]:
from IPython.display import display, FileLink
for item in read_json(PROJECT/'results.json'):
    display(FileLink(item['submission_zip']))
display(FileLink(str(PROJECT/'best_submission.zip')))
ALL_OUTPUTS_ZIP = archive_all(PROJECT)
print('Backup GiB:', round(ALL_OUTPUTS_ZIP.stat().st_size/2**30, 2))
display(FileLink(str(ALL_OUTPUTS_ZIP)))
